# GeoRA 第二步：把一個 Q 適配器接入完整模型

前一份 notebook 已確認單矩陣的初始化與更新。這次要回答：
**完整模型中的第 0 層 Q 換成 GeoRA 後，同一段文字的 logits 是否保持接近？**

這次載入完整 Qwen2.5-1.5B-Instruct，但先只替換一個 Q 模組。
通過後再擴展到全部目標線性層，避免同時引入 196 個替換點。
模型原始 checkpoint 已在本機，程式只讀本機檔案。

## 精度與資源

- 完整模型與 forward 用 CPU FP32，權重約 6.2 GB。
- 這一個 Q 的 mask 與兩次 SVD 繼續用 FP64，方便承接上一份 notebook。
- A、B 轉成 FP32 後，F 以同樣的 FP32 因子計算；轉換及運算誤差會實際測量。

這是本機正確性檢查的選擇。GeoRA 論文附錄 C.1 明寫 BF16 訓練，未明寫 SVD dtype；
PiSSA 官方依賴 PEFT 0.14.0 的 SVD 使用 FP32。
正式 GPU 訓練的精度設定需要另外驗證。

請先關閉上一份 notebook 的 kernel，以釋放它持有的矩陣，然後選這個專案的 `.venv` kernel，從頭逐格執行。
安裝已記錄在 `pyproject.toml` 與 `uv.lock`；在專案目錄也可以用 `uv sync` 還原環境。

## 1. 載入完整模型，辨認目標層

`AutoTokenizer` 把文字轉成 token IDs；`AutoModelForCausalLM` 依本機 config 建立架構並載入權重。
`model.eval()` 關閉訓練時的隨機行為；`requires_grad_(False)` 先凍結全部參數。

論文使用 `all-linear`。這個 Qwen 架構每層有 Q/K/V/O 和 gate/up/down 七個 projection，
28 層共 196 個。本節列出它們，但先只改第 0 層 Q。
`lm_head` 是輸出詞彙分数的模組，這裡不加入適配器。

In [ ]:
from pathlib import Path
from importlib.metadata import version
import gc
import json
import time
import torch
from torch import nn
from torch.nn import functional as functional
from safetensors.torch import save_file, load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_num_threads(4)
project_path = Path.cwd().resolve()
assert (project_path / "geora_layers.py").is_file(), "Run from the repository root."
checkpoint_path = project_path / "checkpoints" / "geora_base"
base_revision = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
model_dtype = torch.float32
r = 16
alpha = 32
rho = 0.2
c = alpha / r

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint_path,
    local_files_only=True,
)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    dtype=model_dtype,
    attn_implementation="eager",
    local_files_only=True,
)
model.eval()
model.requires_grad_(False)

projection_names = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
}
target_names = [
    name for name, module in model.named_modules()
    if isinstance(module, nn.Linear)
    and name.rsplit(".", 1)[-1] in projection_names
]
assert len(target_names) == 28 * 7
assert "lm_head" not in target_names
print("Versions:", {
    package: version(package)
    for package in ["torch", "transformers", "huggingface-hub", "tokenizers"]
})
print("Model dtype:", model.dtype)
print("All projection targets:", len(target_names))
print("First layer targets:", target_names[:7])

## 2. 固定文字與 token IDs，保存原始 logits

chat template 為問題加上這個模型使用的對話標記。
`inputs` 裡包含 `input_ids` 與 `attention_mask`，後續比較都使用同一份。

`logits_before` 的形狀是 `(1, T, vocab_size)`：一段長度 T 的輸入，
每個位置都給整個 vocab 一個分數。這裡比較分數本身，先不做 softmax 或採樣。
只用一個固定短問題檢查程式連接，不測數學能力。

In [ ]:
messages = [{
    "role": "user",
    "content": "What is 2 + 3? Give a short answer.",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
with torch.inference_mode():
    logits_before = model(**inputs, use_cache=False).logits.clone()

assert torch.isfinite(logits_before).all()
print("Input IDs shape:", inputs["input_ids"].shape)
print("Logits shape:", logits_before.shape)

## 3. 把上一份 notebook 的初始化封裝成一個 Q 模組

`initialize_geo_factors` 只負責兩個 mask 與兩次 SVD，回傳 FP64 的 `A0`、`B0`。
`GeoRALinear` 保留原本線性層的 bias，把 weight 改成凍結殘差 F，並新增可訓練的 A、B。

forward 的輸入可以是 `(batch, T, 1536)`，`functional.linear` 對最後一維作用：

$$y=xF^\top+\text{bias}+c(xA^\top)B^\top.$$

這次 FP32 儲存的 A、B 是 FP64 SVD 結果的轉換值。
F 用這些實際儲存的因子計算：`F = W_pre - c * (B @ A)`。
這個 wrapper 是我們的診斷實作，尚未接 PEFT/GRPO 訓練框架。

In [ ]:
@torch.no_grad()
def initialize_geo_factors(weight, rank, sparsity_ratio):
    W_pre = weight.detach().double()
    U_pre, S_pre, Vh_pre = torch.linalg.svd(
        W_pre,
        full_matrices=False,
    )
    W_pre_rank_r = (
        U_pre[:, :rank] * S_pre[:rank]
    ) @ Vh_pre[:rank, :]
    spec_magnitudes = W_pre_rank_r.abs()
    spec_threshold = torch.quantile(
        spec_magnitudes.reshape(-1),
        sparsity_ratio,
        interpolation="linear",
    )
    M_spec = spec_magnitudes <= spec_threshold
    euc_magnitudes = W_pre.abs()
    euc_threshold = torch.quantile(
        euc_magnitudes.reshape(-1),
        sparsity_ratio,
        interpolation="linear",
    )
    M_euc = euc_magnitudes <= euc_threshold
    W_geo = W_pre * (M_spec | M_euc)
    U_geo, S_geo, Vh_geo = torch.linalg.svd(
        W_geo,
        full_matrices=False,
    )
    sqrt_S_geo_r = torch.sqrt(S_geo[:rank])
    B0 = U_geo[:, :rank] * sqrt_S_geo_r
    A0 = sqrt_S_geo_r[:, None] * Vh_geo[:rank, :]
    return A0, B0


class GeoRALinear(nn.Module):
    def __init__(self, base_layer, A_initial, B_initial, scaling):
        super().__init__()
        self.base_layer = base_layer
        self.base_layer.requires_grad_(False)
        self.scaling = scaling
        self.A = nn.Parameter(A_initial.to(base_layer.weight).clone())
        self.B = nn.Parameter(B_initial.to(base_layer.weight).clone())
        with torch.no_grad():
            residual_weight = base_layer.weight - scaling * (self.B @ self.A)
            self.base_layer.weight.copy_(residual_weight)

    def forward(self, x):
        frozen_output = functional.linear(
            x,
            self.base_layer.weight,
            self.base_layer.bias,
        )
        compressed_input = functional.linear(x, self.A)
        adapter_output = functional.linear(compressed_input, self.B)
        return frozen_output + self.scaling * adapter_output

## 4. 原位替換第 0 層 Q，重新跑完整模型

先保存該 Q 的原始權重，用來量測 `F + cBA` 的誤差。
其他層維持原樣，同一個完整模型再跑一次同樣的輸入。

相對誤差用 `差值的 L2 norm / 原值的 L2 norm`；另列最大絕對誤差。
不要求 FP32 兩條計算路徑逐 bit 相等。
權重重建使用 `relative error < 1e-6` 作為 FP32 實作的粗略檢查門檻，這不是論文規定。
logits 的誤差會受後續網路放大，我們報告實際數值與最高分 token，
不把「數值完全一致」作為 FP32 初始化的要求，也不從單個 prompt 推論訓練效果。

In [ ]:
q_name = "model.layers.0.self_attn.q_proj"
original_q = model.get_submodule(q_name)
W_pre_q = original_q.weight.detach().clone()
started = time.perf_counter()
A0_q, B0_q = initialize_geo_factors(
    original_q.weight,
    rank=r,
    sparsity_ratio=rho,
)
geo_q = GeoRALinear(original_q, A0_q, B0_q, scaling=c)
model.model.layers[0].self_attn.q_proj = geo_q
model.eval()

with torch.no_grad():
    W_eff_q = geo_q.base_layer.weight + c * (geo_q.B @ geo_q.A)
    weight_relative_error = (
        torch.linalg.vector_norm((W_eff_q - W_pre_q).double())
        / torch.linalg.vector_norm(W_pre_q.double())
    )
with torch.inference_mode():
    logits_after = model(**inputs, use_cache=False).logits.clone()
    logits_difference = (logits_after - logits_before).double()
    logits_relative_error = (
        torch.linalg.vector_norm(logits_difference)
        / torch.linalg.vector_norm(logits_before.double())
    )
    logits_maximum_error = logits_difference.abs().max()
    last_token_before = logits_before[0, -1].argmax().item()
    last_token_after = logits_after[0, -1].argmax().item()

assert torch.isfinite(logits_after).all()
assert weight_relative_error.item() < 1e-6
print(f"Q effective weight relative error: {weight_relative_error.item():.8e}")
print(f"Full-model logits relative error: {logits_relative_error.item():.8e}")
print(f"Full-model logits maximum absolute error: {logits_maximum_error.item():.8e}")
print("Last-position top token IDs:", last_token_before, last_token_after)
print(f"Initialization + forward seconds: {time.perf_counter() - started:.2f}")

trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
expected_trainable_names = [q_name + ".A", q_name + ".B"]
assert trainable_names == expected_trainable_names
print("Trainable parameters:", trainable_names)
print("Single-Q weight and parameter checks passed; logits error reported above.")

## 5. 儲存／載入這個 Q 模組，確認 logits 保持一致

儲存的 state 包含 **殘差 F、bias、A、B**，另記錄 base revision 與 scaling。
只儲存 A、B 然後加到原始 W 上，會多加一次初始 `cB0A0`，不能這樣重建。

下面重新建立一個 Q wrapper，從檔案載入它的完整 state，替换目前的 Q。
其餘凍結層沿用同一個 model，以減少本機記憶體用量。
這驗證的是「單 Q 模組在完整模型中的 round trip」，尚未驗證全模型 adapter checkpoint。

In [ ]:
artifact_directory = project_path / "outputs" / "geora_single_q_check"
artifact_directory.mkdir(parents=True, exist_ok=True)
state_path = artifact_directory / "q_module.safetensors"
metadata_path = artifact_directory / "metadata.json"

q_state = {
    name: tensor.detach().cpu().contiguous()
    for name, tensor in geo_q.state_dict().items()
}
save_file(q_state, state_path)
metadata = {
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "base_revision": base_revision,
    "module_name": q_name,
    "rank": r,
    "alpha": alpha,
    "rho": rho,
    "model_dtype": "float32",
    "svd_dtype": "float64",
    "attention_implementation": "eager",
    "scope": "single_q_module_in_full_model",
    "weight_relative_error": weight_relative_error.item(),
    "logits_relative_error": logits_relative_error.item(),
    "logits_maximum_absolute_error": logits_maximum_error.item(),
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")

loaded_state = load_file(state_path)
loaded_metadata = json.loads(metadata_path.read_text())
assert loaded_metadata["base_revision"] == base_revision
assert loaded_metadata["module_name"] == q_name
reload_base = nn.Linear(
    original_q.in_features,
    original_q.out_features,
    bias=original_q.bias is not None,
    dtype=model_dtype,
)
reloaded_q = GeoRALinear(
    reload_base,
    loaded_state["A"],
    loaded_state["B"],
    scaling=loaded_metadata["alpha"] / loaded_metadata["rank"],
)
reloaded_q.load_state_dict(loaded_state, strict=True)
model.model.layers[0].self_attn.q_proj = reloaded_q
model.eval()

with torch.inference_mode():
    logits_reloaded = model(**inputs, use_cache=False).logits
reload_maximum_error = (logits_reloaded - logits_after).abs().max().item()
torch.testing.assert_close(logits_reloaded, logits_after, rtol=0, atol=0)
print("Saved Q module:", state_path)
print("Reload logits maximum absolute error:", reload_maximum_error)
print("Single-Q save/reload checks passed.")

## 到這裡完成了什麼？

通過表示：**第 0 層 Q 的 GeoRA wrapper 能接進完整模型；初始化前後 logits 接近；
可訓練參數只有這個 Q 的 A、B；儲存／載入該 Q 模組後，logits 一致。**

我們還需要把同樣流程擴展到全部 196 個 projection，再做整模型參數更新及 checkpoint 檢查。
真正的 GRPO 會接在這些正確性檢查之後。

GRPO reference policy 要使用原始凍結模型，或另外驗證過的初始模型路徑。
單純停用這個 adapter 會留下 F，不能當作原始模型。

完成本節後可關閉 kernel 釋放整模型記憶體；結果已保存在上面的 `outputs` 目錄與本 notebook。

### 來源

- [GeoRA 附錄 C.1：BF16 與 all-linear](https://arxiv.org/html/2601.09361v4)
- [PiSSA 官方訓練腳本：--bf16](https://github.com/MuLabPKU/PiSSA/blob/main/scripts/metamath_llama2_7b/run_pissa.sh)
- [PiSSA 所依賴 PEFT 0.14.0：FP32 SVD](https://github.com/huggingface/peft/blob/v0.14.0/src/peft/tuners/lora/layer.py#L217-L251)